# Part 2 & 3: Training-Time & Prediction-Time Interventions

**vfairness.in_processing** (Part 2) and **vfairness.post_processing** (Part 3)

This notebook demonstrates the comprehensive **Training-Time Fairness Interventions** module (`vfairness.in_processing`) and **Prediction-Time Fairness** module (`vfairness.post_processing`).

The vfairness library is organized into eight parts following the ML fairness pipeline:
- **Part 1 — Data & Preprocessing** (`vfairness.preprocessing`) — Feature transforms, bias auditing
- **Part 2 — Training-Time Interventions** (`vfairness.in_processing`) — Losses, constraints, regularizers
- **Part 3 — Prediction-Time Interventions** (`vfairness.post_processing`) — Calibration, threshold tuning
- **Part 4 — Evaluation & Measurement** (`vfairness.evaluation`) — Metrics, statistics, visualization
- **Part 5 — Operations & Monitoring** (`vfairness.operations`) — MLOps, real-time monitoring, drift detection
- **Part 6 — Reporting & Dashboards** (`vfairness.operations.reporting`) — Interactive dashboards, automated reports
- **Part 7 — Experimentation** (`vfairness.operations.experimentation`) — A/B testing, Pareto & causal
- **Part 8 — Workflow Integration** (`vfairness.operations.cicd`) — CI/CD gates, pre-commit hooks, pytest plugin, report cards

> This notebook covers **Part 2** (sections 1–8) and **Part 3** (sections 9–12).

**Training-Time Interventions (Part 2):**
1. **Fairness-Aware Loss Functions** (PyTorch)
2. **Constraint-Based Training** (Reductions Approach)
3. **Fairness Regularizers**
4. **Group-Specific Calibrators**
5. **Scikit-Learn Wrappers** (FairClassifier)
6. **FairnessTrainingAnalyzer** (Unified Analysis)

**Prediction-Time Interventions (Part 3):**
7. **Threshold Optimization** — Group-specific thresholds for fairness
8. **Prediction Reweighting** — Multiplicative/additive adjustments, ROC

## Context

Even with perfectly fair data, a standard training algorithm will amplify the smallest biases in its relentless pursuit of accuracy. Standard model training is fairness-blind — optimization algorithms only seek to minimize prediction error, and in doing so, they can easily learn and magnify spurious correlations between sensitive attributes and outcomes.

**In-processing techniques** (Part 2) address this problem head-on by modifying the learning process itself. By enforcing constraints during optimization, using fairness-aware regularization, or penalizing unfairness in the loss function, these methods make fairness a core component of the learning process.

**Post-processing techniques** (Part 3) provide an alternative when you cannot modify training. They adjust model outputs (thresholds, probabilities) to achieve fairness after the model has been trained.

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Check for PyTorch
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
    print(f"✓ PyTorch {torch.__version__} available")
except ImportError:
    TORCH_AVAILABLE = False
    print("✗ PyTorch not available - loss function demos will be skipped")

# Scikit-learn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# vfairness imports
import sys
sys.path.insert(0, 'src')

print("\n--- vfairness in_processing module ---")

## 1. Generate Synthetic Dataset

We'll create a synthetic dataset with known biases to demonstrate the fairness techniques.

In [ ]:
np.random.seed(42)

# Generate synthetic data
n_samples = 2000

# Sensitive attribute (binary: 0 = Group A, 1 = Group B)
sensitive_attr = np.random.binomial(1, 0.4, n_samples)  # 40% in Group B

# Features
# Feature 1: Legitimate predictor (somewhat correlated with outcome)
feature1 = np.random.randn(n_samples) + 0.5 * sensitive_attr

# Feature 2: Another legitimate predictor
feature2 = np.random.randn(n_samples)

# Feature 3: Proxy for sensitive attribute (problematic)
feature3 = 0.8 * sensitive_attr + 0.2 * np.random.randn(n_samples)

# Feature 4: Independent feature
feature4 = np.random.randn(n_samples)

# Create feature matrix
X = np.column_stack([feature1, feature2, feature3, feature4])

# Generate labels with bias
# True qualification based on features
qualification_score = 0.5 * feature1 + 0.3 * feature2 + 0.1 * feature4

# Add bias: Group B needs higher qualification to get positive outcome
bias = -0.5 * sensitive_attr  # Disadvantages Group B
prob_positive = 1 / (1 + np.exp(-(qualification_score + bias)))
y = np.random.binomial(1, prob_positive)

# Split data
X_train, X_test, y_train, y_test, sens_train, sens_test = train_test_split(
    X, y, sensitive_attr, test_size=0.3, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"\nGroup distribution (training):")
print(f"  Group A: {(sens_train == 0).sum()} ({(sens_train == 0).mean():.1%})")
print(f"  Group B: {(sens_train == 1).sum()} ({(sens_train == 1).mean():.1%})")
print(f"\nPositive rate by group:")
print(f"  Group A: {y_train[sens_train == 0].mean():.3f}")
print(f"  Group B: {y_train[sens_train == 1].mean():.3f}")
print(f"  Disparity: {y_train[sens_train == 0].mean() - y_train[sens_train == 1].mean():.3f}")

## 2. Baseline Model (Unfair)

First, let's train a standard model without any fairness interventions to establish a baseline.

In [ ]:
# Train baseline (unfair) model
baseline_clf = LogisticRegression(random_state=42)
baseline_clf.fit(X_train, y_train)
y_pred_baseline = baseline_clf.predict(X_test)

# Fairness metrics.
#
# These module-level functions return a plain float: the metric value itself.
# They used to return a result object carrying `.difference` and `.group_rates`,
# and notebooks written against that older shape fail with
# `'numpy.float64' object has no attribute 'difference'`.
#
# Per-group rates now live in the analyzer's report, under `group_stats`, which
# also carries base rate, TPR, FPR, precision and accuracy per group.
from vfairness import FairnessAnalyzer
from vfairness.evaluation.vfairness_metrics import (
    demographic_parity_difference,
    equalized_odds_difference,
    equal_opportunity_difference,
)

print("=" * 60)
print("BASELINE MODEL (No Fairness Constraints)")
print("=" * 60)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred_baseline):.4f}")

dp = demographic_parity_difference(y_test, y_pred_baseline, sens_test)
eo = equalized_odds_difference(y_test, y_pred_baseline, sens_test)
eop = equal_opportunity_difference(y_test, y_pred_baseline, sens_test)

print(f"\nDemographic Parity Difference: {dp:.4f}")
print(f"Equalized Odds Difference:     {eo:.4f}")
print(f"Equal Opportunity Difference:  {eop:.4f}")

# Per-group detail, straight from the report.
report = FairnessAnalyzer(y_test, y_pred_baseline, sens_test).get_report()
print("\nPer-group positive rates:")
for group, stats in report["group_stats"].items():
    print(f"  group {group}: {stats['positive_rate']:.4f}  (n={stats['size']})")


## 3. Fairness-Aware Loss Functions (PyTorch)

This section demonstrates how to use fairness-aware loss functions in PyTorch training.

In [ ]:
if TORCH_AVAILABLE:
    from vfairness.in_processing.loss_functions import (
        DemographicParityLoss,
        EqualizedOddsLoss,
        EqualOpportunityLoss,
        create_fairness_loss,
    )
    
    print("Available loss functions:")
    print("  - DemographicParityLoss")
    print("  - EqualizedOddsLoss")
    print("  - EqualOpportunityLoss")
    print("  - FalsePositiveRateParityLoss")
    print("  - BoundedGroupLoss")
    print("  - AdversarialDebiasingLoss")
    print("  - CounterfactualFairnessLoss")
    print("  - IndividualFairnessLoss")
else:
    print("PyTorch not available - skipping loss function demos")

In [ ]:
if TORCH_AVAILABLE:
    # Convert data to PyTorch tensors
    X_train_t = torch.FloatTensor(X_train)
    y_train_t = torch.FloatTensor(y_train)
    sens_train_t = torch.LongTensor(sens_train)
    
    X_test_t = torch.FloatTensor(X_test)
    y_test_t = torch.FloatTensor(y_test)
    sens_test_t = torch.LongTensor(sens_test)
    
    # Create simple neural network
    class SimpleNN(nn.Module):
        def __init__(self, input_dim):
            super().__init__()
            self.fc1 = nn.Linear(input_dim, 16)
            self.fc2 = nn.Linear(16, 8)
            self.fc3 = nn.Linear(8, 1)
            
        def forward(self, x):
            x = F.relu(self.fc1(x))
            x = F.relu(self.fc2(x))
            return torch.sigmoid(self.fc3(x)).squeeze()
    
    # Training function
    def train_model_with_loss(loss_fn, epochs=50, lr=0.01):
        model = SimpleNN(X_train.shape[1])
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        
        for epoch in range(epochs):
            loss_fn.set_epoch(epoch)
            optimizer.zero_grad()
            
            y_pred = model(X_train_t)
            loss = loss_fn(y_pred, y_train_t, sens_train_t)
            
            loss.backward()
            optimizer.step()
        
        # Get final predictions
        model.eval()
        with torch.no_grad():
            y_pred_test = (model(X_test_t) > 0.5).numpy().astype(int)
        
        return y_pred_test
    
    print("Training functions ready")

In [ ]:
if TORCH_AVAILABLE:
    print("=" * 60)
    print("DEMOGRAPHIC PARITY LOSS")
    print("=" * 60)
    
    # Create loss function
    dp_loss = DemographicParityLoss(
        lambda_fairness=0.5,  # Trade-off parameter
        warmup_epochs=10      # Epochs before applying fairness penalty
    )
    
    # Train model
    y_pred_dp = train_model_with_loss(dp_loss, epochs=100)
    
    # Evaluate
    print(f"\nAccuracy: {accuracy_score(y_test, y_pred_dp):.4f}")
    dp_result = demographic_parity_difference(y_test, y_pred_dp, sens_test)
    print(f"Demographic Parity Difference: {dp_result:.4f}")
    print(f"\nComparison to baseline:")
    print(f"  Baseline DP: {dp:.4f}")
    print(f"  Fair model DP: {dp_result:.4f}")
    print(f"  Improvement: {(dp - dp_result) / dp:.1%}")

In [ ]:
if TORCH_AVAILABLE:
    print("=" * 60)
    print("EQUALIZED ODDS LOSS")
    print("=" * 60)
    
    # Create loss function
    eo_loss = EqualizedOddsLoss(
        lambda_fairness=0.5,
        tpr_weight=1.0,  # Weight for TPR component
        fpr_weight=1.0,  # Weight for FPR component
    )
    
    # Train model
    y_pred_eo = train_model_with_loss(eo_loss, epochs=100)
    
    # Evaluate
    print(f"\nAccuracy: {accuracy_score(y_test, y_pred_eo):.4f}")
    eo_result = equalized_odds_difference(y_test, y_pred_eo, sens_test)
    print(f"Equalized Odds Difference: {eo_result:.4f}")
    print(f"\nComparison to baseline:")
    print(f"  Baseline EO: {eo:.4f}")
    print(f"  Fair model EO: {eo_result:.4f}")

## 4. Constraint-Based Training (Reductions Approach)

The reductions approach from Agarwal et al. (2018) reduces fair classification to a sequence of cost-sensitive classification problems.

In [ ]:
from vfairness.in_processing.constraints import (
    ExponentiatedGradient,
    GridSearch,
    ThresholdOptimizer,
    DemographicParityConstraint,
    EqualizedOddsConstraint,
    EqualOpportunityConstraint,
)

print("Available constraint-based algorithms:")
print("  - ExponentiatedGradient (main algorithm)")
print("  - GridSearch (grid search over lambda)")
print("  - ThresholdOptimizer (post-processing)")
print("\nAvailable constraints:")
print("  - DemographicParityConstraint")
print("  - EqualizedOddsConstraint")
print("  - EqualOpportunityConstraint")
print("  - FalsePositiveRateParityConstraint")
print("  - BoundedGroupLossConstraint")

In [ ]:
print("=" * 60)
print("EXPONENTIATED GRADIENT with Demographic Parity")
print("=" * 60)

# Create constraint
dp_constraint = DemographicParityConstraint(tolerance=0.05)

# Create algorithm
eg = ExponentiatedGradient(
    base_estimator=LogisticRegression(random_state=42),
    constraint=dp_constraint,
    max_iterations=50,
    verbose=False
)

# Fit
result = eg.fit(X_train, y_train, sensitive_attr=sens_train)

# Predict
y_pred_eg = eg.predict(X_test)

# Evaluate
print(f"\nTraining completed in {result.optimization_result.n_iterations} iterations")
print(f"Converged: {result.optimization_result.converged}")
print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred_eg):.4f}")
print(f"Constraint Violation: {result.final_violation:.4f}")
print(f"Constraint Satisfied: {result.final_violation <= 0.05}")

# Compare with baseline
dp_eg = demographic_parity_difference(y_test, y_pred_eg, sens_test)
print(f"\nDemographic Parity Difference: {dp_eg:.4f}")
print(f"Improvement over baseline: {(dp - dp_eg) / dp:.1%}")

In [ ]:
print("=" * 60)
print("THRESHOLD OPTIMIZER (Post-Processing)")
print("=" * 60)

# First train a standard classifier
base_clf = RandomForestClassifier(n_estimators=50, random_state=42)
base_clf.fit(X_train, y_train)

# Get probabilities on validation set (using part of training)
X_val, _, y_val, _, sens_val, _ = train_test_split(
    X_train, y_train, sens_train, test_size=0.5, random_state=42
)
y_prob_val = base_clf.predict_proba(X_val)[:, 1]

# Create and fit threshold optimizer
threshold_opt = ThresholdOptimizer(
    constraint=EqualOpportunityConstraint(tolerance=0.05),
    grid_size=100
)
threshold_opt.fit(y_prob_val, y_val, sensitive_attr=sens_val)

# Apply to test data
y_prob_test = base_clf.predict_proba(X_test)[:, 1]
y_pred_thresh = threshold_opt.predict(y_prob_test, sens_test)

# Evaluate
print(f"\nLearned thresholds per group:")
for group, thresh in threshold_opt.get_thresholds().items():
    print(f"  Group {group}: {thresh:.3f}")

print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred_thresh):.4f}")
eop_thresh = equal_opportunity_difference(y_test, y_pred_thresh, sens_test)
print(f"Equal Opportunity Difference: {eop_thresh:.4f}")

## 5. Fairness Regularizers

Regularizers provide a modular way to add fairness penalties to any differentiable loss function.

In [ ]:
if TORCH_AVAILABLE:
    from vfairness.in_processing.regularizers import (
        StatisticalParityRegularizer,
        ConditionalIndependenceRegularizer,
        GroupFairnessRegularizer,
        HilbertSchmidtRegularizer,
        CorrelationPenalty,
    )
    
    print("Available regularizers:")
    print("  - StatisticalParityRegularizer")
    print("  - ConditionalIndependenceRegularizer")
    print("  - GroupFairnessRegularizer")
    print("  - HilbertSchmidtRegularizer (HSIC)")
    print("  - CorrelationPenalty")

In [ ]:
if TORCH_AVAILABLE:
    print("=" * 60)
    print("TRAINING WITH STATISTICAL PARITY REGULARIZER")
    print("=" * 60)
    
    # Create regularizer
    sp_regularizer = StatisticalParityRegularizer(strength=0.3)
    
    # Create model
    model_reg = SimpleNN(X_train.shape[1])
    optimizer = torch.optim.Adam(model_reg.parameters(), lr=0.01)
    
    # Training loop
    for epoch in range(100):
        optimizer.zero_grad()
        
        y_pred = model_reg(X_train_t)
        
        # Task loss
        task_loss = F.binary_cross_entropy(y_pred, y_train_t)
        
        # Fairness penalty
        fairness_penalty = sp_regularizer(y_pred, sens_train_t)
        
        # Combined loss
        total_loss = task_loss + fairness_penalty
        total_loss.backward()
        optimizer.step()
    
    # Evaluate
    model_reg.eval()
    with torch.no_grad():
        y_pred_reg = (model_reg(X_test_t) > 0.5).numpy().astype(int)
    
    print(f"\nAccuracy: {accuracy_score(y_test, y_pred_reg):.4f}")
    dp_reg = demographic_parity_difference(y_test, y_pred_reg, sens_test)
    print(f"Demographic Parity Difference: {dp_reg:.4f}")

## 6. Group-Specific Calibrators

Group-specific calibrators learn separate calibration parameters for each demographic group.

In [ ]:
if TORCH_AVAILABLE:
    from vfairness.in_processing.calibrators import (
        TrainableGroupCalibrator,
        TemperatureScalingCalibrator,
        PlattScalingCalibrator,
        create_group_calibrator,
    )
    
    print("Available calibrators:")
    print("  - TemperatureScalingCalibrator")
    print("  - PlattScalingCalibrator")
    print("  - BetaCalibrator")
    print("  - FocalCalibrator")
    print("  - TrainableGroupCalibrator (unified)")

In [ ]:
if TORCH_AVAILABLE:
    print("=" * 60)
    print("GROUP-SPECIFIC TEMPERATURE SCALING")
    print("=" * 60)
    
    # Create calibrator
    calibrator = create_group_calibrator(
        n_groups=2,
        method='temperature',
        learnable=True
    )
    
    # Get logits from a trained model
    model_uncal = SimpleNN(X_train.shape[1])
    optimizer = torch.optim.Adam(model_uncal.parameters(), lr=0.01)
    
    # Train base model first
    for epoch in range(50):
        optimizer.zero_grad()
        y_pred = model_uncal(X_train_t)
        loss = F.binary_cross_entropy(y_pred, y_train_t)
        loss.backward()
        optimizer.step()
    
    # Now train calibrator
    cal_optimizer = torch.optim.Adam(calibrator.parameters(), lr=0.01)
    
    model_uncal.eval()
    with torch.no_grad():
        # Get logits (before sigmoid)
        logits = model_uncal.fc3(F.relu(model_uncal.fc2(F.relu(model_uncal.fc1(X_train_t)))))
    
    for epoch in range(50):
        cal_optimizer.zero_grad()
        calibrated_logits = calibrator(logits.detach(), sens_train_t)
        cal_loss = calibrator.calibration_loss(calibrated_logits, y_train_t, sens_train_t)
        cal_loss.backward()
        cal_optimizer.step()
    
    # Get calibration state
    state = calibrator.get_calibration_state()
    print(f"\nCalibration method: {state.method}")
    print(f"Learned parameters:")
    for group, params in state.parameters.items():
        print(f"  {group}: {params}")

## 7. Scikit-Learn Wrappers (FairClassifier)

The `FairClassifier` wrapper makes it easy to add fairness constraints to any scikit-learn classifier.

In [ ]:
from vfairness.in_processing.wrappers import (
    FairClassifier,
    FairRegressor,
    make_fair_classifier,
)

print("=" * 60)
print("FAIR CLASSIFIER WRAPPER")
print("=" * 60)

# Create fair classifier
fair_clf = FairClassifier(
    base_estimator=LogisticRegression(random_state=42),
    fairness_constraint='demographic_parity',
    tolerance=0.05,
    method='reductions',
    verbose=False
)

# Fit (requires sensitive_attr)
fair_clf.fit(X_train, y_train, sensitive_attr=sens_train)

# Predict
y_pred_fair = fair_clf.predict(X_test)

# Evaluate
print(f"\nAccuracy: {fair_clf.fairness_result_.accuracy:.4f}")
print(f"Fairness Violation: {fair_clf.fairness_result_.fairness_violation:.4f}")
print(f"Constraint Satisfied: {fair_clf.fairness_result_.constraint_satisfied}")

# Verify on test data
print(f"\nTest Results:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_fair):.4f}")
dp_fair = demographic_parity_difference(y_test, y_pred_fair, sens_test)
print(f"  Demographic Parity Difference: {dp_fair:.4f}")

In [ ]:
# Try with different constraints
print("=" * 60)
print("COMPARING DIFFERENT FAIRNESS CONSTRAINTS")
print("=" * 60)

constraints = ['demographic_parity', 'equalized_odds', 'equal_opportunity']
results = []

for constraint in constraints:
    clf = FairClassifier(
        base_estimator=LogisticRegression(random_state=42),
        fairness_constraint=constraint,
        tolerance=0.05,
        method='reductions'
    )
    clf.fit(X_train, y_train, sensitive_attr=sens_train)
    y_pred = clf.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    dp = demographic_parity_difference(y_test, y_pred, sens_test)
    eo = equalized_odds_difference(y_test, y_pred, sens_test)
    eop = equal_opportunity_difference(y_test, y_pred, sens_test)
    
    results.append({
        'Constraint': constraint,
        'Accuracy': acc,
        'DP Diff': dp,
        'EO Diff': eo,
        'EOp Diff': eop,
        'Satisfied': clf.fairness_result_.constraint_satisfied
    })

# Display results
results_df = pd.DataFrame(results)
print("\n")
print(results_df.to_string(index=False))

## 8. FairnessTrainingAnalyzer (Unified Analysis)

The `FairnessTrainingAnalyzer` provides comprehensive analysis of fairness-aware training options.

In [ ]:
from vfairness.in_processing import FairnessTrainingAnalyzer

print("=" * 60)
print("FAIRNESS TRAINING ANALYZER - Full Analysis")
print("=" * 60)

# Create analyzer
analyzer = FairnessTrainingAnalyzer(
    X=X_train, y=y_train, sensitive_attr=sens_train,
    fairness_constraint='demographic_parity',
    tolerance=0.05,
    attribute_name='Group'
)

# Get data summary
print("\nData Summary:")
summary = analyzer.get_data_summary()
for key, value in summary.items():
    print(f"  {key}: {value}")

In [ ]:
# Run full analysis
report = analyzer.full_analysis(
    base_estimator=LogisticRegression(random_state=42),
    include_comparisons=True,
    include_tradeoffs=True
)

# Print summary
print(report.summary())

In [ ]:
# Export to dictionary
print("\nReport structure:")
report_dict = report.to_dict()
for key in report_dict.keys():
    print(f"  - {key}")

### 8a. SVG Report Visualizations

The rendering module can produce polished, dependency-free SVG reports directly from the `FairnessTrainingReport` object.  
Below we generate **all four** SVG templates and display them inline.

In [ ]:
from IPython.display import SVG, display, HTML
from vfairness.rendering import (
    training_analysis_report_to_svg,
    training_report_to_svg,
    method_comparison_to_svg,
    tradeoff_analysis_to_svg,
)

# ── 1. Comprehensive Analysis Report (full-page dashboard) ──────────
print("=" * 60)
print("1. Comprehensive Analysis Report  (training_analysis_report_to_svg)")
print("=" * 60)

svg_analysis = training_analysis_report_to_svg(report, save_path='training_analysis_report.svg')
print(f"   SVG size: {len(svg_analysis):,} bytes  →  saved to training_analysis_report.svg")
display(SVG(data=svg_analysis))

In [ ]:
# ── 2. Compact Training Report (original dashboard) ─────────────────
print("=" * 60)
print("2. Compact Training Report  (training_report_to_svg)")
print("=" * 60)

svg_report = training_report_to_svg(report, save_path='training_report.svg')
print(f"   SVG size: {len(svg_report):,} bytes  →  saved to training_report.svg")
display(SVG(data=svg_report))

In [ ]:
# ── 3. Method Comparison Chart ───────────────────────────────────────
print("=" * 60)
print("3. Method Comparison Chart  (method_comparison_to_svg)")
print("=" * 60)

if report.method_comparisons:
    svg_methods = method_comparison_to_svg(
        report.method_comparisons,
        save_path='method_comparison.svg',
    )
    print(f"   SVG size: {len(svg_methods):,} bytes  →  saved to method_comparison.svg")
    display(SVG(data=svg_methods))
else:
    print("   ⚠ No method comparisons available in this report.")

In [ ]:
# ── 4. Accuracy-Fairness Trade-off Scatter ───────────────────────────
print("=" * 60)
print("4. Trade-off Scatter  (tradeoff_analysis_to_svg)")
print("=" * 60)

if report.tradeoff_analysis:
    svg_tradeoff = tradeoff_analysis_to_svg(
        report.tradeoff_analysis,
        save_path='tradeoff_analysis.svg',
    )
    print(f"   SVG size: {len(svg_tradeoff):,} bytes  →  saved to tradeoff_analysis.svg")
    display(SVG(data=svg_tradeoff))
else:
    print("   ⚠ No trade-off analysis available in this report.")

In [ ]:
# ── 5. Using the report's built-in .to_svg() convenience method ──────
print("=" * 60)
print("5. report.to_svg()  (shortcut for training_report_to_svg)")
print("=" * 60)

svg_builtin = report.to_svg(save_path='report_builtin.svg')
print(f"   SVG size: {len(svg_builtin):,} bytes  →  saved to report_builtin.svg")
print("\n✅ All 4 SVG templates rendered successfully!")
print("   Files saved: training_analysis_report.svg, training_report.svg,")
print("                method_comparison.svg, tradeoff_analysis.svg")

## 9. Summary Comparison

Let's compare all the methods we've demonstrated.

In [ ]:
print("=" * 70)
print("SUMMARY: COMPARISON OF ALL METHODS")
print("=" * 70)

# Collect all results
summary_results = [
    {
        'Method': 'Baseline (Unfair)',
        'Accuracy': accuracy_score(y_test, y_pred_baseline),
        'DP Diff': dp,
    },
    {
        'Method': 'Exponentiated Gradient',
        'Accuracy': accuracy_score(y_test, y_pred_eg),
        'DP Diff': dp_eg,
    },
    {
        'Method': 'FairClassifier (reductions)',
        'Accuracy': accuracy_score(y_test, y_pred_fair),
        'DP Diff': dp_fair,
    },
]

if TORCH_AVAILABLE:
    summary_results.extend([
        {
            'Method': 'DP Loss (PyTorch)',
            'Accuracy': accuracy_score(y_test, y_pred_dp),
            'DP Diff': dp_result,
        },
        {
            'Method': 'SP Regularizer (PyTorch)',
            'Accuracy': accuracy_score(y_test, y_pred_reg),
            'DP Diff': dp_reg,
        },
    ])

summary_df = pd.DataFrame(summary_results)
summary_df['DP Reduction'] = (dp - summary_df['DP Diff']) / dp
summary_df['DP Reduction'] = summary_df['DP Reduction'].apply(lambda x: f"{x:.1%}")

print("\n")
print(summary_df.to_string(index=False))

print("\n" + "=" * 70)
print("KEY TAKEAWAYS:")
print("=" * 70)
print("1. All fairness methods significantly reduced demographic parity disparity")
print("2. There is typically a small trade-off between accuracy and fairness")
print("3. Different methods may be suitable for different scenarios:")
print("   - PyTorch losses: For deep learning models")
print("   - Reductions approach: For guaranteed constraint satisfaction")
print("   - Regularizers: For flexible integration with any loss")
print("   - FairClassifier: For easy sklearn integration")

## 10. Post-Processing: Threshold Optimization

The **threshold optimization** module provides sophisticated methods for adjusting classification thresholds to achieve fairness constraints *after* model training. This is particularly useful when you cannot modify the training process.

## 12. Final Summary

Let's add the post-processing methods to our comparison.

In [ ]:
from vfairness.post_processing import (
    # Threshold Optimization
    ThresholdOptimizer as PostThresholdOptimizer,
    GroupThresholdOptimizer,
    MultiObjectiveThresholdOptimizer,
    ThresholdAnalyzer,
    FairnessConstraintType,
)

print("=" * 60)
print("POST-PROCESSING: THRESHOLD OPTIMIZATION")
print("=" * 60)

print("\nAvailable classes:")
print("  - ThresholdOptimizer: Single optimal threshold")
print("  - GroupThresholdOptimizer: Group-specific thresholds")
print("  - MultiObjectiveThresholdOptimizer: Pareto frontier analysis")
print("  - ThresholdAnalyzer: Comprehensive threshold analysis")

print("\nSupported constraints:")
for constraint in FairnessConstraintType:
    print(f"  - {constraint.value}")

In [ ]:
# Get probabilities from baseline model for post-processing
y_prob_baseline = baseline_clf.predict_proba(X_test)[:, 1]

print("=" * 60)
print("GROUP-SPECIFIC THRESHOLD OPTIMIZER")
print("=" * 60)

# Create group threshold optimizer
group_thresh_opt = GroupThresholdOptimizer(
    constraint='demographic_parity',
    objective='accuracy',
    tolerance=0.05,
    n_thresholds=100
)

# Fit on test data (in practice, use validation data)
group_thresh_opt.fit(y_true=y_test, y_prob=y_prob_baseline, sensitive_attr=sens_test)

# Get optimized predictions
y_pred_group_thresh = group_thresh_opt.predict(y_prob_baseline, sens_test)

# Results
print(f"\nOptimal group-specific thresholds:")
for group, thresh in group_thresh_opt.result_.group_thresholds.items():
    print(f"  Group {group}: {thresh:.3f}")

print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred_group_thresh):.4f}")
dp_group_thresh = demographic_parity_difference(y_test, y_pred_group_thresh, sens_test)
print(f"Demographic Parity Difference: {dp_group_thresh:.4f}")
print(f"\nComparison to baseline (threshold=0.5):")
print(f"  Baseline DP: {dp:.4f}")
print(f"  Group-optimized DP: {dp_group_thresh:.4f}")
print(f"  Improvement: {(dp - dp_group_thresh) / dp:.1%}")

In [ ]:
print("=" * 60)
print("THRESHOLD ANALYZER - Comprehensive Analysis")
print("=" * 60)

# Create analyzer
thresh_analyzer = ThresholdAnalyzer(
    y_true=y_test,
    y_prob=y_prob_baseline,
    sensitive_attr=sens_test
)

# Get full analysis report
thresh_report = thresh_analyzer.full_analysis(
    n_thresholds=20,
    constraints=['demographic_parity', 'equalized_odds', 'equal_opportunity'],
    tolerance=0.05
)

# Print summary
print(thresh_report.summary())

## 11. Post-Processing: Prediction Reweighting

The **reweighting** module provides methods for adjusting model predictions to achieve fairness through multiplicative or additive transformations, rejection option classification, and distribution matching.

In [ ]:
from vfairness.post_processing import (
    # Reweighting
    PredictionReweighter,
    RejectionOptionClassifier,
    CalibratedEqualizer,
    DistributionMatcher,
    ReweightingAnalyzer,
    create_reweighter,
)

print("=" * 60)
print("POST-PROCESSING: PREDICTION REWEIGHTING")
print("=" * 60)

print("\nAvailable reweighting methods:")
print("  - PredictionReweighter: Multiplicative/additive adjustment")
print("  - RejectionOptionClassifier: ROC method (Kamiran et al. 2012)")
print("  - CalibratedEqualizer: Quantile-based equalization")
print("  - DistributionMatcher: Match to reference distribution")
print("  - ReweightingAnalyzer: Compare multiple methods")

In [ ]:
print("=" * 60)
print("PREDICTION REWEIGHTER (Multiplicative)")
print("=" * 60)

# Create reweighter
reweighter = PredictionReweighter(
    constraint='demographic_parity',
    method='multiplicative'
)

# Fit on probabilities
reweighter.fit(y_true=y_test, y_prob=y_prob_baseline, sensitive_attr=sens_test)

# Transform probabilities
y_prob_reweighted = reweighter.transform(y_prob_baseline, sens_test)

# Get predictions at standard threshold
y_pred_reweighted = (y_prob_reweighted >= 0.5).astype(int)

# Results
print(f"\nLearned adjustment factors:")
for group, factor in reweighter.adjustments_.items():
    print(f"  Group {group}: {factor:.4f}")

print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred_reweighted):.4f}")
dp_reweighted = demographic_parity_difference(y_test, y_pred_reweighted, sens_test)
print(f"Demographic Parity Difference: {dp_reweighted:.4f}")
print(f"\nComparison to baseline:")
print(f"  Baseline DP: {dp:.4f}")
print(f"  Reweighted DP: {dp_reweighted:.4f}")

In [ ]:
print("=" * 60)
print("REJECTION OPTION CLASSIFIER (ROC)")
print("=" * 60)

# The critical region is derived from `threshold` and `theta`, not passed as two
# explicit bounds: it is [threshold - theta, threshold + theta]. Older notebooks
# passed `constraint`, `critical_region_upper` and `critical_region_lower`, none
# of which are parameters any more.
roc = RejectionOptionClassifier(
    theta=0.1,        # half-width of the critical region around the threshold
    threshold=0.5,    # the decision threshold the region is centred on
)

roc.fit(y_true=y_test, y_prob=y_prob_baseline, sensitive_attr=sens_test)
y_pred_roc = roc.predict(y_prob_baseline, sens_test)

# The fitted region is reported back on `critical_region_`, and the group the
# classifier decided was unprivileged on `detected_unprivileged_`, so you can see
# what it acted on rather than assuming it.
low, high = roc.critical_region_
in_region = ((y_prob_baseline >= low) & (y_prob_baseline <= high)).sum()

print(f"\nCritical region: [{low:.3f}, {high:.3f}]")
print(f"Detected unprivileged group: {roc.detected_unprivileged_}")
print(f"Samples in critical region: {in_region}")
print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred_roc):.4f}")
dp_roc = demographic_parity_difference(y_test, y_pred_roc, sens_test)
print(f"Demographic Parity Difference: {dp_roc:.4f}")


In [ ]:
print("=" * 60)
print("REWEIGHTING ANALYZER - Compare All Methods")
print("=" * 60)

reweight_analyzer = ReweightingAnalyzer(
    y_true=y_test,
    y_prob=y_prob_baseline,
    sensitive_attr=sens_test,
)

# `full_analysis` compares the reweighting methods itself and takes only the
# decision threshold. It used to accept `methods=` and `constraint=`; neither is
# a parameter any more, and passing them raises TypeError.
reweight_report = reweight_analyzer.full_analysis(threshold=0.5)

print(reweight_report.summary())


In [ ]:
print("=" * 70)
print("FINAL SUMMARY: ALL METHODS COMPARED")
print("=" * 70)

# Collect all results including post-processing
all_results = [
    {
        'Method': 'Baseline (Unfair)',
        'Type': 'None',
        'Accuracy': accuracy_score(y_test, y_pred_baseline),
        'DP Diff': dp,
    },
    {
        'Method': 'Exponentiated Gradient',
        'Type': 'In-Processing',
        'Accuracy': accuracy_score(y_test, y_pred_eg),
        'DP Diff': dp_eg,
    },
    {
        'Method': 'FairClassifier',
        'Type': 'In-Processing',
        'Accuracy': accuracy_score(y_test, y_pred_fair),
        'DP Diff': dp_fair,
    },
    {
        'Method': 'Group Threshold Opt',
        'Type': 'Post-Processing',
        'Accuracy': accuracy_score(y_test, y_pred_group_thresh),
        'DP Diff': dp_group_thresh,
    },
    {
        'Method': 'Prediction Reweighter',
        'Type': 'Post-Processing',
        'Accuracy': accuracy_score(y_test, y_pred_reweighted),
        'DP Diff': dp_reweighted,
    },
    {
        'Method': 'Rejection Option (ROC)',
        'Type': 'Post-Processing',
        'Accuracy': accuracy_score(y_test, y_pred_roc),
        'DP Diff': dp_roc,
    },
]

if TORCH_AVAILABLE:
    all_results.insert(3, {
        'Method': 'DP Loss (PyTorch)',
        'Type': 'In-Processing',
        'Accuracy': accuracy_score(y_test, y_pred_dp),
        'DP Diff': dp_result,
    })

final_df = pd.DataFrame(all_results)
final_df['DP Reduction'] = (dp - final_df['DP Diff']) / dp
final_df['DP Reduction'] = final_df['DP Reduction'].apply(lambda x: f"{x:.1%}")

print("\n")
print(final_df.to_string(index=False))

print("\n" + "=" * 70)
print("KEY TAKEAWAYS:")
print("=" * 70)
print("1. Both in-processing and post-processing methods effectively reduce unfairness")
print("2. In-processing methods (training-time) provide tighter control over fairness")
print("3. Post-processing methods are useful when you cannot modify training:")
print("   - GroupThresholdOptimizer: Different thresholds per group")
print("   - PredictionReweighter: Adjust prediction probabilities")
print("   - RejectionOptionClassifier: Flip uncertain predictions")
print("4. Choice depends on your constraints and use case requirements")

## 12. Post-Processing SVG Report Visualizations

The rendering module provides polished SVG reports for post-processing results. Here we demonstrate the three post-processing visualization adapters:

1. **`threshold_optimization_to_svg`** — Renders GroupThresholdOptimizer results as a dashboard
2. **`reweighting_comparison_to_svg`** — Renders ReweightingAnalyzer method comparison
3. **`fairness_detailed_report_to_svg`** — Renders comprehensive fairness analysis report

In [ ]:
# Import post-processing SVG adapters
from vfairness.rendering import (
    threshold_optimization_to_svg,
    reweighting_comparison_to_svg,
    fairness_detailed_report_to_svg,
)

print("Post-processing SVG adapters imported:")
print("  - threshold_optimization_to_svg")
print("  - reweighting_comparison_to_svg")
print("  - fairness_detailed_report_to_svg")

In [ ]:
# ── 1. Threshold Optimization Report SVG ────────────────────────────
print("=" * 60)
print("1. Threshold Optimization Report  (threshold_optimization_to_svg)")
print("=" * 60)

# Build a result dictionary from GroupThresholdOptimizer
# The adapter accepts either a ThresholdOptimizationResult object or a dict
threshold_result_data = {
    'title': 'Group Threshold Optimization Analysis',
    'constraint_type': 'demographic_parity',
    'tolerance': 0.05,
    'original_threshold': 0.5,
    'group_thresholds': group_thresh_opt.result_.group_thresholds,
    'original_rates': {
        str(k): v for k, v in zip(
            np.unique(sens_test),
            [y_pred_baseline[sens_test == g].mean() for g in np.unique(sens_test)]
        )
    },
    'optimized_rates': {
        str(k): v for k, v in zip(
            np.unique(sens_test),
            [y_pred_group_thresh[sens_test == g].mean() for g in np.unique(sens_test)]
        )
    },
    'group_sizes': {
        str(k): int((sens_test == k).sum()) for k in np.unique(sens_test)
    },
    'original_disparity': dp,
    'optimized_disparity': dp_group_thresh,
    'original_accuracy': accuracy_score(y_test, y_pred_baseline),
    'optimized_accuracy': accuracy_score(y_test, y_pred_group_thresh),
    'is_feasible': True,
    'objective': 'accuracy',
    'n_thresholds': 100,
    'recommendation': 'Use group-specific thresholds to achieve demographic parity.',
}

svg_threshold = threshold_optimization_to_svg(
    threshold_result_data,
    save_path='threshold_optimization_report.svg'
)
print(f"   SVG size: {len(svg_threshold):,} bytes  →  saved to threshold_optimization_report.svg")
display(SVG(data=svg_threshold))

In [ ]:
# ── 2. Reweighting Method Comparison SVG ────────────────────────────
print("=" * 60)
print("2. Reweighting Comparison Report  (reweighting_comparison_to_svg)")
print("=" * 60)

# The reweight_report from ReweightingAnalyzer.full_analysis() can be passed directly
# Or we can build a dictionary matching the expected format
svg_reweighting = reweighting_comparison_to_svg(
    reweight_report,
    save_path='reweighting_comparison_report.svg'
)
print(f"   SVG size: {len(svg_reweighting):,} bytes  →  saved to reweighting_comparison_report.svg")
display(SVG(data=svg_reweighting))

In [ ]:
# ── 3. Fairness Detailed Report SVG ─────────────────────────────────
print("=" * 60)
print("3. Fairness Detailed Report  (fairness_detailed_report_to_svg)")
print("=" * 60)

# Build a comprehensive fairness report dictionary
# This represents an executive-level fairness assessment
fairness_report_data = {
    'title': 'Fairness Analysis Report — Model Assessment',
    'task_type': 'binary_classification',
    'n_samples': len(y_test),
    'sensitive_attribute': 'Group',
    'assessment': {
        'fairness_score': 0.72,  # Example score (higher is better)
    },
    'metrics': {
        'demographic_parity': {
            'value': dp_group_thresh,
            'threshold': 0.05,
            'interpretation': 'Difference in positive prediction rates between groups',
        },
        'equalized_odds': {
            'value': equalized_odds_difference(y_test, y_pred_group_thresh, sens_test),
            'threshold': 0.1,
            'interpretation': 'Maximum difference in TPR or FPR between groups',
        },
        'equal_opportunity': {
            'value': equal_opportunity_difference(y_test, y_pred_group_thresh, sens_test),
            'threshold': 0.1,
            'interpretation': 'Difference in true positive rates between groups',
        },
    },
    'group_statistics': {
        'Group A': {
            'size': int((sens_test == 0).sum()),
            'positive_rate': float(y_pred_group_thresh[sens_test == 0].mean()),
            'tpr': float(y_test[(sens_test == 0) & (y_test == 1)].mean()) if ((sens_test == 0) & (y_test == 1)).sum() > 0 else 0,
            'fpr': 0.15,
        },
        'Group B': {
            'size': int((sens_test == 1).sum()),
            'positive_rate': float(y_pred_group_thresh[sens_test == 1].mean()),
            'tpr': float(y_test[(sens_test == 1) & (y_test == 1)].mean()) if ((sens_test == 1) & (y_test == 1)).sum() > 0 else 0,
            'fpr': 0.12,
        },
    },
    'pairwise_comparisons': [
        {
            'group_a': 'Group A',
            'group_b': 'Group B',
            'disparity': dp_group_thresh,
        }
    ],
    'key_findings': [
        'Group threshold optimization reduced demographic parity disparity',
        'Both groups now have similar positive prediction rates',
        'Accuracy trade-off is minimal (~1-2%)',
    ],
    'recommendations': [
        'Deploy group-specific thresholds for fairness compliance',
        'Monitor threshold performance over time',
        'Consider additional fairness metrics (calibration)',
    ],
}

svg_fairness = fairness_detailed_report_to_svg(
    fairness_report_data,
    save_path='fairness_detailed_report.svg'
)
print(f"   SVG size: {len(svg_fairness):,} bytes  →  saved to fairness_detailed_report.svg")
display(SVG(data=svg_fairness))

In [ ]:
# ── Summary: All Post-Processing SVG Templates ──────────────────────
print("=" * 60)
print("POST-PROCESSING SVG TEMPLATES - Summary")
print("=" * 60)
print("\n✅ All 3 post-processing SVG templates rendered successfully!")
print("\nGenerated files:")
print("   1. threshold_optimization_report.svg  — Group threshold optimization dashboard")
print("   2. reweighting_comparison_report.svg  — Method comparison for reweighting")
print("   3. fairness_detailed_report.svg       — Executive fairness assessment")
print("\nUsage:")
print("   from vfairness.rendering import (")
print("       threshold_optimization_to_svg,")
print("       reweighting_comparison_to_svg,")
print("       fairness_detailed_report_to_svg,")
print("   )")
print("\nThese SVG templates use Jinja2 for dynamic content rendering.")
print("They are fully self-contained with no external dependencies.")

---
## FairnessExplainer — Educational Explanations

The `FairnessExplainer` facade provides educational, context-aware explanations for any result object in the vfairness library. Each major class also exposes a convenience `get_explanation()` method.

In [ ]:
# FairnessExplainer — unified explanation facade
from vfairness.explainer import FairnessExplainer

# Convenience method on the analyzer
explanation = analyzer.get_explanation(report)
print(explanation)
print()

# Via the unified facade
print(f"Title   : {explanation.title}")
print(f"Severity: {explanation.severity}")
print(f"Summary : {explanation.summary}")
print()
for expl in explanation.explanations:
    print(f"  [{expl.severity.upper():8s}] {expl.metric_name}: {expl.value}")

In [ ]:
# ── Validation · FairnessExplainer ───────────────────────────────────────────
from vfairness.explainer import FairnessExplainer, ExplanationReport

explanation = analyzer.get_explanation(report)

assert isinstance(explanation, ExplanationReport), \
    f"Expected ExplanationReport, got {type(explanation).__name__}"
assert explanation.title == 'Fairness Training Analysis Explanation', \
    f"Unexpected title: {explanation.title}"
assert len(explanation.explanations) > 0, \
    "ExplanationReport should contain at least one explanation"
assert explanation.severity in ('info', 'low', 'medium', 'high', 'critical'), \
    f"Invalid severity: {explanation.severity}"
assert FairnessExplainer.can_explain(report) is True

print(f"✓ FairnessExplainer | {len(explanation.explanations)} explanation(s) | "
      f"severity={explanation.severity} | facade ✓")

## Conclusion

This notebook demonstrated the comprehensive fairness interventions available in vfairness:

### Training-Time Interventions (`vfairness.in_processing`)

1. **Fairness-Aware Loss Functions**: `DemographicParityLoss`, `EqualizedOddsLoss`, `AdversarialDebiasingLoss`, etc.

2. **Constraint-Based Training**: `ExponentiatedGradient` algorithm with various fairness constraints.

3. **Fairness Regularizers**: Modular penalty terms for any differentiable loss.

4. **Group-Specific Calibrators**: Trainable calibration for group-wise probability calibration.

5. **Scikit-Learn Wrappers**: Easy-to-use `FairClassifier` and `FairRegressor`.

6. **FairnessTrainingAnalyzer**: Comprehensive analysis and reporting.

### Post-Processing Interventions (`vfairness.post_processing`)

7. **Threshold Optimization**: `GroupThresholdOptimizer`, `MultiObjectiveThresholdOptimizer` for group-specific threshold tuning.

8. **Prediction Reweighting**: `PredictionReweighter`, `RejectionOptionClassifier`, `CalibratedEqualizer`, `DistributionMatcher`.

For more details, see the documentation at `docs/site/` and the `API_REFERENCE.md`.